# Token 缓冲记忆

> **将对话历史裁剪到严格的 token 预算内。使用真正的 tokenizer，因为消息数量是上下文窗口使用量的糟糕代理。**

想象打包一个严格限重的行李箱。你不会数物品的数量——你称每件物品的重量，然后移除最旧的物品直到所有东西都能装下。Token 缓冲记忆以同样的方式工作：它按 token（而非消息数量）衡量对话历史，并移除最旧的消息直到总量符合预算。

滑动窗口记忆（技术 02）基于固定数量丢弃消息。但一条包含代码块的消息可能消耗数千 token，而简短的"yes"只消耗一个。Token 缓冲记忆用精确测量取代了这种粗略计数。**tokenizer** 是将文本分割为 **token**（模型内部处理的词片）的工具。集成一个像 `tiktoken` 这样的库。现在系统确切知道历史在到达模型之前消耗了多少 token。

这种精确性在生产中最为重要。过大的提示词要么默默截断，要么抛出硬错误。过小的提示词浪费昂贵的上下文窗口空间。Token 缓冲记忆消除了这两种失败模式。它设置硬上限（`max_token_limit`）并在总量超过时驱逐最旧的消息。

本技术刻意不花哨：没有摘要，没有嵌入（文本的向量表示），没有检索。这种简单性就是它的优势——它是确定性的、快速的、易于理解的。当你需要更复杂的保留策略时，将它分层叠加摘要或检索增强记忆。Token 缓冲记忆仍然是保持提示词在边界内的基础护栏。

**完成本 notebook 后你将理解：**
- Tokenizer 如何测量真实的上下文使用量，以及为什么 token 计数优于消息计数。
- 如何使用 `tiktoken` 和 OpenAI SDK 从头构建 token 感知记忆。
- Token 预算何时闪耀，何时你需要更强大的方案。

## 核心概念

- **Token**：语言模型读取的最小单位。大约一个词或词片。单词 "unhappiness" 可能分成两个 token："un" 和 "happiness"。
- **Tokenizer（`tiktoken`）**：将文本分割为特定模型 token 的库。OpenAI 的 `tiktoken` 处理 GPT-4、GPT-4o 及相关模型。
- **编码**：模型使用的特定 token 词汇表。GPT-4 使用 `cl100k_base`，GPT-4o 使用 `o200k_base`。错误的编码会给出不准确的计数。
- **max_token_limit**：对话历史允许的总 token 硬上限。系统驱逐旧消息直到总量保持在此数值或以下。
- **每条消息的开销**：每条聊天消息除了内容本身外还有格式化 token。这些包括角色标记、内容分隔符和消息结束分隔符。通常每条消息 4 个 token。
- **消息级驱逐**：整条消息被移除（而非部分裁剪）。这保持了连贯性，但可能略微未充分利用预算。
- **上下文窗口**：模型在单次调用中能处理的最大 token 数。GPT-4o 支持 128K token。你的历史预算必须为系统提示词和模型回复留出空间。

## 架构

<p align="center">
  <img src="../../images/diagrams/05_token_buffer_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid 源代码</summary>

```mermaid
flowchart LR
    NewMsg["新消息"] --> Tokenizer["Tokenizer\n(tiktoken)"]
    Tokenizer --> Counter["Token 计数器\n(汇总所有消息)"]
    Counter --> BudgetCheck{"总量 > \nmax_token_limit？"}
    BudgetCheck -- 否 --> TrimmedHistory["裁剪后的历史"]
    BudgetCheck -- 是 --> Evict["驱逐最旧\n消息"]
    Evict --> Counter
    TrimmedHistory --> LLM["LLM"]
    LLM --> Response["回复"]
    Response --> NewMsg
```

</details>

**数据流：** 每条新消息被 tokenize。计数器累加历史中的所有消息。如果总量超过 `max_token_limit`，最旧的消息被驱逐并在循环中重新检查计数。一旦低于预算，裁剪后的历史发送给 LLM。回复被追加，循环重复。

## 环境准备

安装依赖并配置 API 访问。

In [ ]:
%pip install -q tiktoken openai python-dotenv

导入 `tiktoken` 进行 token 计数，导入 OpenAI SDK 进行 LLM 调用。API 密钥从 `.env` 文件加载。

In [ ]:
import os

import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # 从 .env 读取 OPENAI_API_KEY

client = OpenAI()
MODEL = "gpt-4o"
ENCODING = tiktoken.encoding_for_model(MODEL)

assert os.getenv("OPENAI_API_KEY"), "请在 .env 文件中设置 OPENAI_API_KEY"

## 实现

我们将构建一个 `TokenBufferMemory` 类，它：
1. 使用 `tiktoken` 统计每条消息的 token。
2. 维护所有存储消息的运行总数。
3. 当总量超过 `max_token_limit` 时驱逐最旧的消息。
4. 每次调用时将裁剪后的历史发送给 OpenAI API。

### Token 计数

准确的 token 计数是本技术的基础。每条聊天消息消耗的 token 比其内容本身多。Chat Completions API 用特殊 token 包装每条消息。这些包括角色标记、内容分隔符和消息结束分隔符。这个开销通常为每条消息 4 个 token。

让我们看看 `tiktoken` 如何为不同长度的消息计数 token。

In [ ]:
def count_message_tokens(message: dict) -> int:
    """计算单条聊天消息的 token 数，包含每条消息的开销。

    开销（每条消息 4 个 token）：
      <|im_start|> + 角色 + 内容分隔符 + <|im_end|>
    """
    overhead = 4
    return overhead + len(ENCODING.encode(message["content"]))


# 演示：计算不同长度消息的 token 数
sample_messages = [
    {"role": "user", "content": "yes"},
    {"role": "user", "content": "法国的首都是什么？"},
    {"role": "user", "content": (
        "编写一个 Python 函数，使用带记忆化的动态规划计算第 n 个斐波那契数。"
    )},
]

for msg in sample_messages:
    tokens = count_message_tokens(msg)
    print(f"  {tokens:3d} token | \"{msg['content'][:70]}\"")

### TokenBufferMemory 类

该类将 token 计数封装成一个完整的记忆系统。在每次 `chat()` 调用时，它：
1. 追加用户消息。
2. 累加所有存储消息的 token。
3. 循环驱逐最旧的消息直到总量符合 `max_token_limit`。
4. 将裁剪后的历史发送给 LLM。
5. 追加回复。

将 `max_token_limit` 设置为可用于历史的空间。从模型的上下文窗口中减去系统提示词 token 和你的回复预算。例如，对于 128K 窗口、200 token 的系统提示词和 1,024 token 的回复预算，将 `max_token_limit` 设置为约 126,000。

In [ ]:
class TokenBufferMemory:
    """具有严格 token 预算的 token 感知对话记忆。"""

    def __init__(
        self,
        model: str = "gpt-4o",
        max_token_limit: int = 2048,
        system_prompt: str | None = None,
        max_response_tokens: int = 512,
    ):
        self.client = OpenAI()
        self.model = model
        self.max_token_limit = max_token_limit
        self.system_prompt = system_prompt
        self.max_response_tokens = max_response_tokens

        # 为目标模型选择正确的编码
        self.encoding = tiktoken.encoding_for_model(model)

        # 核心数据结构：消息缓冲区
        self.messages: list[dict] = []

        # 跟踪驱逐以进行检查
        self.eviction_log: list[dict] = []

    # -- Token 计数 -----------------------------------------------
    def count_tokens(self, message: dict) -> int:
        """计算单条消息的 token 数，包含每条消息的开销。"""
        overhead = 4  # <|im_start|>、角色、分隔符、<|im_end|>
        return overhead + len(self.encoding.encode(message["content"]))

    def total_tokens(self) -> int:
        """汇总所有已存储消息的 token 数。"""
        return sum(self.count_tokens(m) for m in self.messages)



现在我们添加驱逐循环和 `chat` 方法。`_trim` 方法运行一个 while 循环：它弹出最旧的消息，减去其 token 数，重复直到总量符合 `max_token_limit`。`chat` 方法追加用户消息、必要时裁剪、调用 LLM，并追加回复。

In [ ]:
    # -- 驱逐循环 ------------------------------------------------
    def _trim(self) -> None:
        """移除最旧的消息，直到总 token 数符合预算。"""
        total = self.total_tokens()
        while total > self.max_token_limit and len(self.messages) > 1:
            removed = self.messages.pop(0)
            removed_tokens = self.count_tokens(removed)
            total -= removed_tokens
            self.eviction_log.append({
                "role": removed["role"],
                "preview": removed["content"][:60],
                "tokens_freed": removed_tokens,
            })

    # -- 对话 ---------------------------------------------------------
    def chat(self, user_input: str) -> str:
        """发送消息，超出预算时裁剪，返回回复。"""
        self.messages.append({"role": "user", "content": user_input})
        self._trim()

        # 构建 API 请求体
        api_messages = []
        if self.system_prompt:
            api_messages.append({"role": "system", "content": self.system_prompt})
        api_messages.extend(self.messages)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=api_messages,
            max_tokens=self.max_response_tokens,
        )

        assistant_text = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_text})

        return assistant_text



用于检查和重置记忆的工具方法。`get_messages` 返回当前历史的副本。`clear` 清空消息和驱逐日志。

In [ ]:
    # -- 工具方法 ----------------------------------------------------
    def get_messages(self) -> list[dict]:
        """返回消息历史的副本。"""
        return [m.copy() for m in self.messages]

    def clear(self) -> None:
        """重置所有消息和日志。"""
        self.messages.clear()
        self.eviction_log.clear()

    def __len__(self) -> int:
        return len(self.messages)

    def __repr__(self) -> str:
        return (
            f"TokenBufferMemory("
            f"{len(self.messages)} 条消息, "
            f"{self.total_tokens()}/{self.max_token_limit} token)"
        )

## 示例运行

一次简短对话展示 token 缓冲记忆的实际运作。使用慷慨的预算（2,048 token），所有消息都能装下，没有被驱逐的内容。Agent 记住了一切。

In [ ]:
memory = TokenBufferMemory(
    model="gpt-4o",
    max_token_limit=2048,
    system_prompt="你是一个有帮助、简洁的助手。回复请控制在两句话以内。",
)

exchanges = [
    "你好！我叫 Alice，是一名机器学习工程师。",
    "我正在构建一个关于 Agent 记忆系统的项目。",
    "我叫什么名字，我在做什么？",  # 回忆测试
]

for msg in exchanges:
    print(f"用户：  {msg}")
    reply = memory.chat(msg)
    print(f"Agent：{reply}")
    print(f"       [{memory.total_tokens()} / {memory.max_token_limit} token, "
          f"{len(memory)} 条消息]\n")

### 驱逐的实际演示

现在我们使用一个极小的预算（300 token）。这会在几轮后强制驱逐。观察随着旧消息消失，token 计数如何保持在边界内。

In [ ]:
small_memory = TokenBufferMemory(
    model="gpt-4o",
    max_token_limit=300,  # 故意设置得很小以触发驱逐
    system_prompt="请用一句简短的话回复。",
)

demo_messages = [
    "我叫 Bob，住在东京。",
    "我有两只猫，名叫 Mochi 和 Sushi。",
    "我最喜欢的语言是 Rust。",
    "我在一家机器人初创公司工作，有 20 名工程师。",
    "你记得关于我的什么？",  # 一些早期的事实将丢失
]

for msg in demo_messages:
    tokens_before = small_memory.total_tokens()
    reply = small_memory.chat(msg)
    tokens_after = small_memory.total_tokens()
    print(f"用户：  {msg}")
    print(f"Agent：{reply}")
    print(f"       [token: {tokens_before} -> {tokens_after}, "
          f"消息数: {len(small_memory)}]\n")

# 显示被驱逐的内容
if small_memory.eviction_log:
    print("--- 驱逐日志 ---")
    for entry in small_memory.eviction_log:
        print(f"  已移除 ({entry['role']}): "
              f"\"{entry['preview']}\" "
              f"（释放 {entry['tokens_freed']} token）")

### 为什么 Token 优于消息计数

一个保留最后 5 条消息的滑动窗口将所有消息视为平等。实际上，消息在大小上可能相差几个数量级。一个单词的回复和一个粘贴的代码块在滑动窗口中都算作"一条消息"。但它们在 token 数量上差异巨大。

下面的代码单元直接展示了这一点。

In [ ]:
comparison_messages = [
    {"role": "user", "content": "是的"},
    {"role": "assistant", "content": "没问题！"},
    {"role": "user", "content": (
        "这是我的完整配置文件：\n"
        "import logging\n"
        "import os\n"
        "from pathlib import Path\n\n"
        "BASE_DIR = Path(__file__).resolve().parent\n"
        "SECRET_KEY = os.environ['SECRET_KEY']\n"
        "DEBUG = os.environ.get('DEBUG', 'False') == 'True'\n"
        "ALLOWED_HOSTS = os.environ.get('ALLOWED_HOSTS', '').split(',')\n"
        "DATABASES = {\n"
        "    'default': {\n"
        "        'ENGINE': 'django.db.backends.postgresql',\n"
        "        'NAME': os.environ['DB_NAME'],\n"
        "        'USER': os.environ['DB_USER'],\n"
        "        'PASSWORD': os.environ['DB_PASSWORD'],\n"
        "        'HOST': os.environ.get('DB_HOST', 'localhost'),\n"
        "        'PORT': os.environ.get('DB_PORT', '5432'),\n"
        "    }\n"
        "}\n"
        "LOGGING = {\n"
        "    'version': 1,\n"
        "    'handlers': {\n"
        "        'console': {'class': 'logging.StreamHandler'},\n"
        "    },\n"
        "    'root': {'handlers': ['console'], 'level': 'WARNING'},\n"
        "}\n"
    )},
]

print("消息计数 vs. token 计数：\n")
total_tokens = 0
for i, msg in enumerate(comparison_messages):
    tokens = count_message_tokens(msg)
    total_tokens += tokens
    preview = msg["content"][:50].replace("\n", " ")
    suffix = "..." if len(msg["content"]) > 50 else ""
    print(f"  消息 {i + 1} ({msg['role']:>9}): {tokens:4d} token "
          f"| \"{preview}{suffix}\"")

print(f"\n  总计：{total_tokens} token，共 {len(comparison_messages)} 条消息")
print(f"\n  3 条消息的滑动窗口会保留全部三条。")
print(f"  50 token 的预算会丢弃消息 3 "
      f"（{count_message_tokens(comparison_messages[2])} token）。")
print(f"  基于 token 的裁剪在这里做出了正确的选择。")

## 权衡

### Token 缓冲记忆适用场景
- **精确的预算控制**：你确切知道有多少 token 发给模型。没有默默截断，没有拒绝的提示词。
- **可变消息大小**：代码块、JSON 负载和简短回复都能准确测量。消息计数裁剪做不到这一点。
- **速度**：驱逐在毫秒内运行，无需 LLM 调用（不同于摘要记忆）。裁剪是确定性的。
- **多模型路由**：如果你的系统路由到具有不同上下文窗口的模型，为每个模型设置 `max_token_limit`。同一个类处理所有情况。

### 失效场景
- **无法回忆被驱逐的内容**：一旦消息被移除，该信息就消失了。摘要记忆或检索增强记忆可以保留部分内容。
- **一条大消息挤掉许多小消息**：一条 1,000 token 的消息会迫使驱逐许多短消息以腾出空间。
- **Tokenizer 必须匹配模型**：使用 GPT-4 的编码配合 GPT-4o（或反之）会得到错误的计数。保持编码与模型同步。
- **整消息驱逐可能浪费预算**：如果预算是 500 token 而最旧的消息消耗 400，驱逐它释放的空间超过了需要。部分裁剪会更高效，但会破坏消息连贯性。

### 下一步？
- **[06：向量存储记忆](../06_vector_store_memory/)** 使用语义搜索检索相关历史消息，而非按时间丢弃。
- **[03：摘要记忆](../03_summary_memory/)** 将旧消息压缩为摘要，使信息不完全丢失。
- 组合方案：使用 token 缓冲作为硬上限，配合摘要或检索来保留被驱逐的内容。

## 进一步阅读

- [LangChain ConversationTokenBufferMemory](https://python.langchain.com/docs/modules/memory/types/token_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：LangChain 的内置实现。它封装 tiktoken 进行自动基于 token 的裁剪。
- [tiktoken：OpenAI 的快速 BPE Tokenizer](https://github.com/openai/tiktoken)：OpenAI 模型的参考 tokenizer。
- [OpenAI 实用指南：如何使用 tiktoken 计数 Token](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：准确计数 token 的逐步指南。涵盖 Chat Completions API 的每条消息开销。
- [Anthropic Token 计数 API](https://docs.anthropic.com/en/docs/build-with-claude/token-counting?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：Anthropic 的 Claude 模型服务端 token 计数端点。

---

*← 上一章：[04 摘要缓冲记忆](../04_summary_buffer_memory/) · 下一章：[06 向量存储记忆](../06_vector_store_memory/) →*

## 🧪 自己动手试试

三个小挑战来加深你的理解。每个应该花费 10-30 分钟。

### 挑战 1：基于优先级的裁剪
修改 `_trim()` 使其为每条消息打分（例如，包含问题或名称的消息获得更高优先级），并优先驱逐优先级最低的消息，而非最旧的消息。运行 15 轮对话，比较其回忆表现与默认的 FIFO 方案。

### 挑战 2：预算扫描
使用 `max_token_limit` 设置为 500、1000、2000 和 4000 运行相同的对话。对每种预算，记录保留的消息数、被驱逐的消息数，以及 Agent 正确回答了多少回忆问题。以表格形式呈现结果。

### 挑战 3：带摘要溢出的 Token 预算
当 `_trim()` 驱逐消息时，将它们传递给摘要器而非直接丢弃。将摘要作为系统提示词的补充前置。测量每轮使用的总 token 并与纯 token 缓冲对比。这将 05 Token 缓冲与 03 摘要记忆的方法结合。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--05-token-buffer-memory--token-buffer-memory)
